# IE-OPT-05 · 排列流水车间生产调度（原创 PySCIPOpt 教学案例）

四个工件需按同一个顺序依次经过三台机器，已知各工件在各机器上的处理时长，目标是最小化最大完工时间（makespan）。

参考 [PySCIPOpt 官方 pfs.py](https://github.com/scipopt/PySCIPOpt/blob/master/examples/finished/pfs.py)，但本站使用独立编写的工件位置建模和枚举核验。

**与本站 IE-DES-03 的区别**：这里是静态确定性、同序三机流水车间，默认无限工序间等待；IE-DES-03 是随机到达、有限缓冲、会发生加工后阻塞的双工序离散事件仿真。不能混用两个模型的指标。

## 1. 建模

- x_(j,k) ∈ {0,1}：工件 j 是否排在全局次序的第 k 个位置。
- C_(m,k)：第 k 个位置的工件完成第 m 工序的时间。
- 同位置的工件必须先完成上道工序，同台机器必须按位置先后完成任务。
- C_(m,k) ≥ C_(m-1,k) + Σ_j p_(j,m)x_(j,k)；C_(m,k) ≥ C_(m,k-1) + Σ_j p_(j,m)x_(j,k)。
- 最小化 C_(最后一台机器,最后一个工件位置)。

In [ ]:
from itertools import permutations
from pyscipopt import Model, quicksum

processing = [
    [2, 5, 3],  # 工件一在机器 1/2/3 上的工时
    [4, 2, 4],
    [3, 3, 1],
    [5, 2, 2]
]
n = len(processing)
m = len(processing[0])
assert all(len(row) == m for row in processing)

In [ ]:
def solve_permutation_flowshop(processing):
    n, m = len(processing), len(processing[0])
    model = Model("IE-OPT-05 permutation flowshop")
    model.hideOutput(True)
    x = {(j, k): model.addVar(vtype="B", name=f"position_{j}_{k}")
         for j in range(n) for k in range(n)}
    completion = {(machine, k): model.addVar(lb=0, name=f"end_{machine}_{k}")
                  for machine in range(m) for k in range(n)}
    for j in range(n):
        model.addCons(quicksum(x[j, k] for k in range(n)) == 1)
    for k in range(n):
        model.addCons(quicksum(x[j, k] for j in range(n)) == 1)
    for machine in range(m):
        for k in range(n):
            duration = quicksum(processing[j][machine] * x[j, k]
                                for j in range(n))
            if machine > 0:
                model.addCons(completion[machine, k] >=
                              completion[machine - 1, k] + duration)
            if k > 0:
                model.addCons(completion[machine, k] >=
                              completion[machine, k - 1] + duration)
            if machine == 0 and k == 0:
                model.addCons(completion[machine, k] >= duration)
    model.setObjective(completion[m - 1, n - 1], "minimize")
    model.optimize()
    assert str(model.getStatus()) == "optimal"
    order = [next(j for j in range(n) if model.getVal(x[j, k]) > .5)
             for k in range(n)]
    return {"makespan": float(model.getObjVal()), "order": order}

answer = solve_permutation_flowshop(processing)
print("最优 makespan:", answer["makespan"])
print("工件顺序（0 起）:", answer["order"])

## 2. 独立精确验证

只有 4 个工件，共 4! = 24 种排列。完全不调用 SCIP，按物理加工顺序计算每个排列的完工矩阵，对照 MIP 最优值。

In [ ]:
def evaluate_schedule(order, processing):
    n, m = len(order), len(processing[0])
    completion = [[0] * n for _ in range(m)]
    for machine in range(m):
        for position, job in enumerate(order):
            machine_available = completion[machine][position - 1] if position else 0
            job_available = completion[machine - 1][position] if machine else 0
            completion[machine][position] = max(machine_available, job_available) + processing[job][machine]
    return completion[-1][-1]

oracle = min(evaluate_schedule(order, processing)
             for order in permutations(range(n)))
assert abs(answer["makespan"] - oracle) < 1e-6
assert sorted(answer["order"]) == list(range(n))
assert evaluate_schedule(answer["order"], processing) == oracle
print("24 种排列穷举通过，Cmax =", oracle)

## 3. 从排程走向真实系统仿真

1. 增加或减少某台机器的加工工时，重求最优排程；哪个工序更容易成为瓶颈？
2. 如果工序间缓冲区为 0，当前无限缓冲的理论排程是否仍可执行？请使用本站双工序生产线仿真验证阻塞现象。
3. 如果加工时间存在误差，请区分排程阶段已知信息与执行阶段才观测到的随机工时。
4. 本模型不处理工人技能、AGV、返工、设备故障或动态插单。禁止把这些未模拟资源的利用率写成结果。

**本 Notebook 为可验证课堂例题，不是已上线 L2 私有评分任务。**